In [11]:
# ============================================================
# TATA POWER EV CHARGING ANALYTICS
# STAGE 5 — EXPERIMENT DESIGN
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Project paths
PROJECT_ROOT = Path.cwd().parent

DATA_RAW_PATH = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLE_PATH = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURE_PATH = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_TABLE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_PATH.mkdir(parents=True, exist_ok=True)

print("Stage 5 — Experiment Design")
print("=" * 60)
print("Project root:", PROJECT_ROOT)
print("Random seed:", RANDOM_SEED)

Stage 5 — Experiment Design
Project root: /Users/trinetradhararg/tata-power-ev-charging-analytics
Random seed: 42


In [12]:
# ============================================================
# 1. LOAD INTERVENTION-ELIGIBLE CUSTOMER DATA
# ============================================================

customer_priority_path = (
    OUTPUT_TABLE_PATH / "intervention_customer_priority.csv"
)

customer_priority = pd.read_csv(customer_priority_path)

print("Customers loaded:", len(customer_priority))
print("\nColumns:")
print(customer_priority.columns.tolist())

print("\nEligible customers:")
print(
    customer_priority["eligible_for_intervention"]
    .value_counts(dropna=False)
)

Customers loaded: 8000

Columns:
['customer_id', 'customer_segment_x', 'total_sessions', 'peak_sessions', 'peak_share', 'total_energy_kwh', 'avg_energy_kwh', 'flexibility_score', 'eligible_for_intervention', 'intervention_score', 'priority_band']

Eligible customers:
eligible_for_intervention
True     4661
False    3339
Name: count, dtype: int64


In [13]:
# ============================================================
# 2. DEFINE EXPERIMENT POPULATION
# ============================================================

# Load city information from the original customer master
customers_master = pd.read_csv(
    DATA_RAW_PATH / "customers.csv"
)

print("Customer master columns:")
print(customers_master.columns.tolist())

# Keep only the fields we need
customer_city = customers_master[
    ["customer_id", "city"]
].drop_duplicates("customer_id")

# Add city to Stage 4 intervention population
eligible_customers = (
    customer_priority[
        customer_priority["eligible_for_intervention"] == True
    ]
    .copy()
    .merge(
        customer_city,
        on="customer_id",
        how="left"
    )
    .reset_index(drop=True)
)

print("\nExperiment population:", len(eligible_customers))

print("\nMissing city values:")
print(eligible_customers["city"].isna().sum())

print("\nSegment distribution:")
print(
    eligible_customers["customer_segment_x"]
    .value_counts()
)

print("\nTop cities:")
print(
    eligible_customers["city"]
    .value_counts()
    .head(10)
)

Customer master columns:
['customer_id', 'city', 'customer_segment', 'battery_capacity_kwh', 'home_charging_access', 'preferred_station_type', 'base_sessions_per_month', 'peak_preference_score', 'price_sensitivity_score', 'convenience_score', 'station_loyalty_score', 'baseline_avg_session_kwh', 'incentive_response_score']

Experiment population: 4661

Missing city values:
0

Segment distribution:
customer_segment_x
Routine_Commuter     2041
Convenience_First    1186
Price_Sensitive       972
High_Mileage          462
Name: count, dtype: int64

Top cities:
city
Delhi NCR    580
Bengaluru    569
Mumbai       537
Pune         438
Hyderabad    437
Chennai      436
Ahmedabad    325
Kolkata      316
Lucknow      283
Jaipur       255
Name: count, dtype: int64


In [14]:
# ============================================================
# 3. RANDOMISED TREATMENT / CONTROL ASSIGNMENT
# ============================================================

eligible_customers["random_value"] = rng.random(
    len(eligible_customers)
)

eligible_customers["experiment_group"] = np.where(
    eligible_customers["random_value"] < 0.50,
    "Treatment",
    "Control"
)

eligible_customers["experiment_group"].value_counts()

experiment_group
Treatment    2357
Control      2304
Name: count, dtype: int64

In [15]:
# ============================================================
# 4. RANDOMISATION BALANCE CHECK
# ============================================================

balance_check = (
    eligible_customers
    .groupby("experiment_group")
    .agg(
        customers=("customer_id", "count"),
        avg_peak_share=("peak_share", "mean"),
        avg_flexibility=("flexibility_score", "mean"),
        avg_total_sessions=("total_sessions", "mean"),
        avg_peak_sessions=("peak_sessions", "mean"),
        avg_energy_kwh=("total_energy_kwh", "mean")
    )
    .round(3)
)

print(balance_check)

                  customers  avg_peak_share  avg_flexibility  \
experiment_group                                               
Control                2304           0.338           65.124   
Treatment              2357           0.337           65.155   

                  avg_total_sessions  avg_peak_sessions  avg_energy_kwh  
experiment_group                                                         
Control                      136.319             45.518        3445.511  
Treatment                    136.025             45.385        3425.823  


In [16]:
# ============================================================
# 5. DEFINE INTERVENTION
# ============================================================

# Off-peak incentive offered to treatment customers
INCENTIVE_RATE = 0.15

# Target off-peak window
OFF_PEAK_START = 0
OFF_PEAK_END = 6

eligible_customers["incentive_rate"] = np.where(
    eligible_customers["experiment_group"] == "Treatment",
    INCENTIVE_RATE,
    0.0
)

eligible_customers["intervention"] = np.where(
    eligible_customers["experiment_group"] == "Treatment",
    "Off-Peak Incentive",
    "No Incentive"
)

print("Intervention design")
print("=" * 60)
print(f"Incentive rate: {INCENTIVE_RATE:.0%}")
print(f"Target window: {OFF_PEAK_START:02d}:00–{OFF_PEAK_END:02d}:00")
print("\nIntervention assignment:")
print(
    eligible_customers[
        ["experiment_group", "intervention"]
    ].value_counts()
)

Intervention design
Incentive rate: 15%
Target window: 00:00–06:00

Intervention assignment:
experiment_group  intervention      
Treatment         Off-Peak Incentive    2357
Control           No Incentive          2304
Name: count, dtype: int64


In [17]:
# ============================================================
# 6. SIMULATE INTERVENTION RESPONSE
# ============================================================

# Segment-specific probability of shifting a peak session
response_probability = {
    "High_Mileage": 0.45,
    "Routine_Commuter": 0.35,
    "Price_Sensitive": 0.30,
    "Convenience_First": 0.20
}

eligible_customers["base_response_probability"] = (
    eligible_customers["customer_segment_x"]
    .map(response_probability)
    .fillna(0.25)
)

# More flexible customers should respond more strongly.
flexibility_adjustment = (
    (eligible_customers["flexibility_score"] - 50) / 100
)

eligible_customers["response_probability"] = (
    eligible_customers["base_response_probability"]
    + 0.20 * flexibility_adjustment
).clip(0.05, 0.75)

# Only treatment customers receive the intervention.
eligible_customers["responded"] = False

treatment_mask = (
    eligible_customers["experiment_group"] == "Treatment"
)

eligible_customers.loc[
    treatment_mask,
    "responded"
] = (
    rng.random(treatment_mask.sum())
    < eligible_customers.loc[
        treatment_mask,
        "response_probability"
    ]
)

print("Treatment response rate:")
print(
    eligible_customers.loc[
        treatment_mask,
        "responded"
    ].mean()
)

Treatment response rate:
0.3559609673313534


In [18]:
# ============================================================
# 7. GENERATE POST-INTERVENTION OUTCOMES
# ============================================================

# Customers who respond shift a portion of their peak sessions.
# The amount varies by flexibility.

shift_fraction = (
    0.25
    + 0.50
    * (
        eligible_customers["flexibility_score"] / 100
    )
)

eligible_customers["shifted_peak_sessions"] = np.where(
    eligible_customers["responded"],
    np.round(
        eligible_customers["peak_sessions"]
        * shift_fraction
    ),
    0
)

# Cannot shift more sessions than the customer actually has.
eligible_customers["shifted_peak_sessions"] = (
    eligible_customers[
        "shifted_peak_sessions"
    ]
    .clip(
        lower=0,
        upper=eligible_customers["peak_sessions"]
    )
    .astype(int)
)

eligible_customers["post_peak_sessions"] = (
    eligible_customers["peak_sessions"]
    - eligible_customers["shifted_peak_sessions"]
)

eligible_customers["demand_shift_rate"] = (
    eligible_customers["shifted_peak_sessions"]
    / eligible_customers["peak_sessions"].replace(0, np.nan)
).fillna(0)

print(
    eligible_customers[
        [
            "customer_id",
            "experiment_group",
            "customer_segment_x",
            "peak_sessions",
            "shifted_peak_sessions",
            "post_peak_sessions",
            "demand_shift_rate"
        ]
    ].head(10)
)

  customer_id experiment_group customer_segment_x  peak_sessions  \
0   CUST01186          Control       High_Mileage            172   
1   CUST03288        Treatment       High_Mileage            201   
2   CUST05705          Control       High_Mileage            161   
3   CUST06073          Control       High_Mileage            159   
4   CUST03013        Treatment       High_Mileage            201   
5   CUST06920          Control       High_Mileage            148   
6   CUST05223          Control       High_Mileage            155   
7   CUST05568          Control       High_Mileage            179   
8   CUST04791        Treatment       High_Mileage            162   
9   CUST00082        Treatment       High_Mileage            151   

   shifted_peak_sessions  post_peak_sessions  demand_shift_rate  
0                      0                 172           0.000000  
1                      0                 201           0.000000  
2                      0                 161         

In [19]:
# ============================================================
# 8. EXPERIMENT OUTCOME SUMMARY
# ============================================================

experiment_summary = (
    eligible_customers
    .groupby("experiment_group")
    .agg(
        customers=("customer_id", "count"),
        responders=("responded", "sum"),
        shifted_peak_sessions=("shifted_peak_sessions", "sum"),
        peak_sessions=("peak_sessions", "sum")
    )
)

experiment_summary["response_rate"] = (
    experiment_summary["responders"]
    / experiment_summary["customers"]
)

experiment_summary["demand_shift_rate"] = (
    experiment_summary["shifted_peak_sessions"]
    / experiment_summary["peak_sessions"]
)

print(experiment_summary.round(4))

                  customers  responders  shifted_peak_sessions  peak_sessions  \
experiment_group                                                                
Control                2304           0                      0         104873   
Treatment              2357         839                  25361         106972   

                  response_rate  demand_shift_rate  
experiment_group                                    
Control                   0.000             0.0000  
Treatment                 0.356             0.2371  


In [20]:
# ============================================================
# 9. SAVE EXPERIMENT DATA
# ============================================================

experiment_output = eligible_customers.drop(
    columns=["random_value"],
    errors="ignore"
)

experiment_path = (
    OUTPUT_TABLE_PATH / "experiment_assignments.csv"
)

experiment_output.to_csv(
    experiment_path,
    index=False
)

print(f"Experiment dataset saved:")
print(experiment_path)
print(f"Rows: {len(experiment_output):,}")

Experiment dataset saved:
/Users/trinetradhararg/tata-power-ev-charging-analytics/outputs/tables/experiment_assignments.csv
Rows: 4,661
